# Notebook "Teste d'abord" pour tests Python avec pytest

Ce notebook présente une approche test-first pour écrire et exécuter des tests Python dans le projet `DevGLPI`.

- Créer un environnement de test local
- Charger le code du projet
- Écrire des tests unitaires et d'intégration
- Exécuter `pytest` depuis le notebook
- Générer un rapport de couverture
- Ajouter un workflow GitHub Actions

## 1. Importer les bibliothèques requises

Importez `pytest`, `importlib`, `pathlib`, `sys` et toute bibliothèque utilitaire nécessaire.

Si `pytest` n’est pas installé, utilisez `!pip install pytest`.

In [ ]:
import importlib
import pathlib
import sys

print('Python version:', sys.version)
print('Current cwd:', pathlib.Path.cwd())

## 2. Charger le code du projet dans le notebook

Ajoutez le dossier du projet à `sys.path`, rechargez les modules avec `importlib.reload`, et importez une fonction cible à tester.

In [ ]:
PROJECT_ROOT = pathlib.Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

import backend.repositories.ticket_repository as ticket_repository
importlib.reload(ticket_repository)

from backend.repositories.ticket_repository import _resolve_project_id

print('Loaded _resolve_project_id:', _resolve_project_id)

## 3. Configurer l'environnement de test (pytest)

Créez un fichier de configuration `pytest.ini` minimal et configurez les options de test.

In [ ]:
pytest_ini = PROJECT_ROOT / 'pytest.ini'
pytest_ini.write_text('[pytest]
addopts = -q
python_files = test_*.py
')
print('Created', pytest_ini)

## 4. Écrire un premier test unitaire (test-first)

Créez un fichier de test simple `backend/tools/test_example.py` contenant un test qui échoue d’abord, puis implémentez la fonction minimale.

In [ ]:
test_path = PROJECT_ROOT / 'backend' / 'tools' / 'test_example.py'
test_path.write_text(
    'def test_example_failure():
'
    '    assert 1 + 1 == 2
'
)
print('Created', test_path)

## 5. Tests paramétriques et jeux de données

Utilisez `@pytest.mark.parametrize` pour tester plusieurs entrées et résultats attendus.

In [ ]:
param_test_path = PROJECT_ROOT / 'backend' / 'tools' / 'test_project_resolution_param.py'
param_test_path.write_text(
    'import pytest
'
    'from backend.repositories.ticket_repository import _resolve_project_id

'
    '@pytest.mark.parametrize(
'
    '    "ticket,expected",
'
    '    [
'
    '        ({"projects_id": 5}, 5),
'
    '        ({"project_id": "7"}, 7),
'
    '        ({"project": {"id": 8}}, 8),
'
    '        ({"projects": "10"}, 10),
'
    '        ({}, 0),
'
    '    ],
'
    ')
'
    'def test_resolve_project_id(ticket, expected):
'
    '    assert _resolve_project_id(ticket) == expected
'
)

## 6. Tester les cas d’erreur et exceptions

Écrivez des tests avec `pytest.raises` pour valider les exceptions ou les entrées invalides.

In [ ]:
error_test_path = PROJECT_ROOT / 'backend' / 'tools' / 'test_project_resolution_errors.py'
error_test_path.write_text(
    'import pytest
'
    'from backend.repositories.ticket_repository import _resolve_project_id

'
    'def test_resolve_project_id_with_invalid_type():
'
    '    assert _resolve_project_id({"project": ["abc"]}) == 0
'
)

## 7. Tests d’intégration simples

Écrivez un test d’intégration qui combine plusieurs modules pour vérifier le flux complet.

In [ ]:
integration_test_path = PROJECT_ROOT / 'backend' / 'tools' / 'test_integration_ticket_enrichment.py'
integration_test_path.write_text(
    'from backend.repositories.ticket_repository import TicketRepository
'
    'from backend.core.config import Settings
'
    'from backend.clients.mock_client import MockClient

'
    'def test_ticket_enrichment_with_mock_client():
'
    '    settings = Settings(use_mock_data=False)
'
    '    repo = TicketRepository(settings, MockClient(settings))
'
    '    tickets = [
'
    '        {"id": 1, "content": "<b>1) Projet</b>: M001 -- DN & Administration"},
'
    '    ]
'
    '    enriched = repo._enrich(tickets)
'
    '    assert enriched[0]["_project_name"] == "M001 -- DN & Administration"
'
)

## 8. Exécuter les tests depuis le notebook

Montrez comment lancer `pytest` via `!pytest -q` ou `%%bash pytest`, capturer la sortie et afficher les résultats.

In [ ]:
!pytest -q backend/tools/test_example.py backend/tools/test_project_resolution_param.py backend/tools/test_project_resolution_errors.py backend/tools/test_integration_ticket_enrichment.py

## 9. Générer et afficher un rapport de couverture

Installez `coverage` et exécutez `pytest --cov=backend --cov-report=term-missing`.

In [ ]:
!pip install coverage pytest --quiet
!pytest --cov=backend --cov-report=term-missing backend/tools/test_project_resolution_param.py

## 10. Ajouter un workflow GitHub Actions pour exécuter les tests

Créez `.github/workflows/ci.yml` avec un job installé pour exécuter `pytest`.

In [ ]:
ci_path = PROJECT_ROOT / '.github' / 'workflows' / 'ci.yml'
ci_path.parent.mkdir(parents=True, exist_ok=True)
ci_path.write_text(
    'name: CI
'
    'on: [push, pull_request]
'
    'jobs:
'
    '  test:
'
    '    runs-on: ubuntu-latest
'
    '    steps:
'
    '      - uses: actions/checkout@v4
'
    '      - name: Setup Python
'
    '        uses: actions/setup-python@v5
'
    '        with:
'
    '          python-version: 3.12
'
    '      - name: Install dependencies
'
    '        run: pip install pytest coverage
'
    '      - name: Run tests
'
    '        run: pytest -q
'
    '      - name: Coverage
'
    '        run: pytest --cov=backend --cov-report=term-missing
'
)
print('Created', ci_path)

# Notebook "Teste d'abord" pour tests Python avec pytest

Ce notebook présente une approche test-first pour écrire et exécuter des tests Python dans le projet `DevGLPI`.

- Créer un environnement de test local
- Charger le code du projet
- Écrire des tests unitaires et d'intégration
- Exécuter `pytest` depuis le notebook
- Générer un rapport de couverture
- Ajouter un workflow GitHub Actions

## 1. Importer les bibliothèques requises

Importez `pytest`, `importlib`, `pathlib`, `sys` et toute bibliothèque utilitaire nécessaire.

Si `pytest` n’est pas installé, utilisez `!pip install pytest`.

In [ ]:
import importlib
import pathlib
import sys

print('Python version:', sys.version)
print('Current cwd:', pathlib.Path.cwd())

## 2. Charger le code du projet dans le notebook

Ajoutez le dossier du projet à `sys.path`, rechargez les modules avec `importlib.reload`, et importez une fonction cible à tester.

In [ ]:
PROJECT_ROOT = pathlib.Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

import backend.repositories.ticket_repository as ticket_repository
importlib.reload(ticket_repository)

from backend.repositories.ticket_repository import _resolve_project_id

print('Loaded _resolve_project_id:', _resolve_project_id)

## 3. Configurer l'environnement de test (pytest)

Créez un fichier de configuration `pytest.ini` minimal et configurez les options de test.

In [ ]:
pytest_ini = PROJECT_ROOT / 'pytest.ini'
pytest_ini.write_text('[pytest]
addopts = -q
python_files = test_*.py
')
print('Created', pytest_ini)

## 4. Écrire un premier test unitaire (test-first)

Créez un fichier de test simple `backend/tools/test_example.py` contenant un test qui échoue d’abord, puis implémentez la fonction minimale.

In [ ]:
test_path = PROJECT_ROOT / 'backend' / 'tools' / 'test_example.py'
test_path.write_text(
    'def test_example_failure():
'
    '    assert 1 + 1 == 2
'
)
print('Created', test_path)

## 5. Tests paramétriques et jeux de données

Utilisez `@pytest.mark.parametrize` pour tester plusieurs entrées et résultats attendus.

In [ ]:
param_test_path = PROJECT_ROOT / 'backend' / 'tools' / 'test_project_resolution_param.py'
param_test_path.write_text(
    'import pytest
'
    'from backend.repositories.ticket_repository import _resolve_project_id

'
    '@pytest.mark.parametrize(
'
    '    "ticket,expected",
'
    '    [
'
    '        ({"projects_id": 5}, 5),
'
    '        ({"project_id": "7"}, 7),
'
    '        ({"project": {"id": 8}}, 8),
'
    '        ({"projects": "10"}, 10),
'
    '        ({}, 0),
'
    '    ],
'
    ')
'
    'def test_resolve_project_id(ticket, expected):
'
    '    assert _resolve_project_id(ticket) == expected
'
)

## 6. Tester les cas d’erreur et exceptions

Écrivez des tests avec `pytest.raises` pour valider les exceptions ou les entrées invalides.

In [ ]:
error_test_path = PROJECT_ROOT / 'backend' / 'tools' / 'test_project_resolution_errors.py'
error_test_path.write_text(
    'import pytest
'
    'from backend.repositories.ticket_repository import _resolve_project_id

'
    'def test_resolve_project_id_with_invalid_type():
'
    '    assert _resolve_project_id({"project": ["abc"]}) == 0
'
)

## 7. Tests d’intégration simples

Écrivez un test d’intégration qui combine plusieurs modules pour vérifier le flux complet.

In [ ]:
integration_test_path = PROJECT_ROOT / 'backend' / 'tools' / 'test_integration_ticket_enrichment.py'
integration_test_path.write_text(
    'from backend.repositories.ticket_repository import TicketRepository
'
    'from backend.core.config import Settings
'
    'from backend.clients.mock_client import MockClient

'
    'def test_ticket_enrichment_with_mock_client():
'
    '    settings = Settings(use_mock_data=False)
'
    '    repo = TicketRepository(settings, MockClient(settings))
'
    '    tickets = [
'
    '        {"id": 1, "content": "<b>1) Projet</b>: M001 -- DN & Administration"},
'
    '    ]
'
    '    enriched = repo._enrich(tickets)
'
    '    assert enriched[0]["_project_name"] == "M001 -- DN & Administration"
'
)

## 8. Exécuter les tests depuis le notebook

Montrez comment lancer `pytest` via `!pytest -q` ou `%%bash pytest`, capturer la sortie et afficher les résultats.

In [ ]:
!pytest -q backend/tools/test_example.py backend/tools/test_project_resolution_param.py backend/tools/test_project_resolution_errors.py backend/tools/test_integration_ticket_enrichment.py

## 9. Générer et afficher un rapport de couverture

Installez `coverage` et exécutez `pytest --cov=backend --cov-report=term-missing`.

In [ ]:
!pip install coverage pytest --quiet
!pytest --cov=backend --cov-report=term-missing backend/tools/test_project_resolution_param.py

## 10. Ajouter un workflow GitHub Actions pour exécuter les tests

Créez `.github/workflows/ci.yml` avec un job installé pour exécuter `pytest`.

In [ ]:
ci_path = PROJECT_ROOT / '.github' / 'workflows' / 'ci.yml'
ci_path.parent.mkdir(parents=True, exist_ok=True)
ci_path.write_text(
    'name: CI
'
    'on: [push, pull_request]
'
    'jobs:
'
    '  test:
'
    '    runs-on: ubuntu-latest
'
    '    steps:
'
    '      - uses: actions/checkout@v4
'
    '      - name: Setup Python
'
    '        uses: actions/setup-python@v5
'
    '        with:
'
    '          python-version: 3.12
'
    '      - name: Install dependencies
'
    '        run: pip install pytest coverage
'
    '      - name: Run tests
'
    '        run: pytest -q
'
    '      - name: Coverage
'
    '        run: pytest --cov=backend --cov-report=term-missing
'
)
print('Created', ci_path)